# Yellow Taxi ingestion
Run this notebook first in Microsoft Fabric with a default Lakehouse attached. Place the 24 Parquet files in `Files/raw/yellow_taxi` and the zone CSV in `Files/reference/taxi_zone_lookup.csv`. The write step creates `bronze_trips` once. For a fresh run, use a Lakehouse where that table does not already exist.

In [ ]:
%%configure -f
{
  "conf": {
    "spark.native.enabled": "false",
    "spark.gluten.sql.native.writer.enabled": "false",
    "spark.sql.cbo.enabled": "false",
    "spark.sql.cbo.planStats.enabled": "false"
  }
}

## Verify the monthly files

In [ ]:
# Check that all 24 monthly Parquet files are present.

folder_path = "Files/raw/yellow_taxi"

expected_files = [
    f"yellow_tripdata_{year}-{month:02d}.parquet"
    for year in (2022, 2023)
    for month in range(1, 13)
]

parquet_items = sorted(
    [
        item
        for item in notebookutils.fs.ls(folder_path)
        if item.isFile and item.name.lower().endswith(".parquet")
    ],
    key=lambda item: item.name
)

actual_files = [item.name for item in parquet_items]
# The set difference lists expected file names that were absent from the folder.
missing_files = sorted(set(expected_files) - set(actual_files))
unexpected_files = sorted(set(actual_files) - set(expected_files))

print(f"Parquet files found: {len(parquet_items)}")
print(f"Missing files: {missing_files}")
print(f"Unexpected files: {unexpected_files}")

# Stop here if the input folder does not match the two complete study years.
if missing_files or unexpected_files:
    raise ValueError(f"Check the monthly inputs. Missing: {missing_files}. Unexpected: {unexpected_files}.")

for item in parquet_items:
    size_mb = item.size / (1024 * 1024)
    print(f"{item.name} | {size_mb:.1f} MB")

In [ ]:
# Check raw Parquet files and group them by schema.
from collections import defaultdict

# Use the exact file list checked above.
file_items = parquet_items

schema_groups = defaultdict(list)

for item in file_items:
    file_path = f"{folder_path}/{item.name}"
    schema = spark.read.parquet(file_path).schema

    # The ordered pairs of column names and types form a key. Files with the same key share a schema.
    schema_signature = tuple(
        (field.name, field.dataType.simpleString())
        for field in schema.fields
    )

    schema_groups[schema_signature].append(item.name)

print(f"Files inspected: {len(file_items)}")
print(f"Distinct schema versions: {len(schema_groups)}")

for version, (signature, file_names) in enumerate(schema_groups.items(), start=1):
    print(f"\nSchema version {version}: {len(file_names)} file(s)")
    print("Files:", ", ".join(file_names))

    for column_name, data_type in signature:
        print(f"  {column_name}: {data_type}")

In [ ]:
# Combine the monthly files after aligning their schemas.
from functools import reduce
from pyspark.sql import functions as F

# Use the exact file list checked above.
file_items = parquet_items

monthly_frames = []

for item in file_items:
    file_path = f"{folder_path}/{item.name}"
    # This is the file month. Pickup dates are used later for report trends.
    source_month = item.name.replace("yellow_tripdata_", "").replace(".parquet", "")

    monthly_df = spark.read.parquet(file_path)

    # Standardise the fee name and numeric types before stacking the monthly rows.
    if "Airport_fee" in monthly_df.columns:
        monthly_df = monthly_df.withColumnRenamed("Airport_fee", "airport_fee")

    monthly_df = (
        monthly_df
        .withColumn("VendorID", F.col("VendorID").cast("long"))
        .withColumn("passenger_count", F.col("passenger_count").cast("double"))
        .withColumn("RatecodeID", F.col("RatecodeID").cast("double"))
        .withColumn("PULocationID", F.col("PULocationID").cast("long"))
        .withColumn("DOLocationID", F.col("DOLocationID").cast("long"))
        .withColumn("payment_type", F.col("payment_type").cast("long"))
        .withColumn("source_file", F.lit(item.name))
        .withColumn("source_month", F.lit(source_month))
    )

    monthly_frames.append(monthly_df)

# Start with the first monthly frame and add the next one each time. unionByName matches columns by name even if their order differs.
combined_df = reduce(
    lambda left, right: left.unionByName(right),
    monthly_frames
)

print(f"Files combined: {len(monthly_frames)}")
combined_df.printSchema()

combined_df.select(
    "source_month",
    "source_file",
    "VendorID",
    "tpep_pickup_datetime",
    "PULocationID",
    "airport_fee"
).limit(5).show(truncate=False)

In [ ]:
# Save the combined rows as the Bronze Delta table.
bronze_table_name = "bronze_trips"

if spark.catalog.tableExists(bronze_table_name):
    raise ValueError(
        f"Table {bronze_table_name} already exists. "
        "Inspect it before deciding whether to reuse or replace it."
    )

(
    combined_df.write
    .format("delta")
    .mode("errorifexists")
    .saveAsTable(bronze_table_name)
)

print(f"Created Delta table: {bronze_table_name}")

In [ ]:
# Count rows for each monthly source file.
from pyspark.sql import functions as F

bronze_df = spark.table("bronze_trips")

# Spark counts rows by source file first. Only the small monthly summary is collected into Python.
monthly_counts = (
    bronze_df
    .groupBy("source_month", "source_file")
    .agg(F.count("*").alias("row_count"))
    .orderBy("source_month")
    .collect()
)

print(f"Source files counted: {len(monthly_counts)}")

for row in monthly_counts:
    print(
        f"{row['source_month']} | "
        f"{row['source_file']} | "
        f"{row['row_count']:,} rows"
    )

# A file with no rows would be missing from the grouped result.
counted_files = {row["source_file"] for row in monthly_counts}
if counted_files != set(expected_files):
    raise ValueError("Bronze does not contain rows from every expected source file.")

total_rows = sum(row["row_count"] for row in monthly_counts)
print(f"Total Bronze rows: {total_rows:,}")

In [ ]:
# Load the taxi zone reference file for location checks.
from pyspark.sql import functions as F

zone_lookup_path = "Files/reference/taxi_zone_lookup.csv"

zone_lookup_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(zone_lookup_path)
)

print("Taxi zone lookup schema:")
zone_lookup_df.printSchema()

print("Sample rows:")
zone_lookup_df.show(5, truncate=False)

## Recorded results for the submitted dataset
- Study period: January 2022 to December 2023
- Monthly Parquet files loaded: 24
- Bronze table: bronze_trips
- Total Bronze records: 77,966,324
- Source lineage columns: source_file and source_month
- Schema differences were harmonized before combining the files.
- No rows were removed during ingestion.